
# AdaBoost Malware Detection with Balanced Error Rate

This notebook trains and evaluates AdaBoost models on the `AndroidmalwareSmall.xlsx` dataset (10,000 Android applications) with the goal of minimizing the balanced error rate (BER). The focus is on identifying a model configuration that performs well on future, similarly structured data.



## 1. Setup and Data Loading

Read the dataset and review its structure to confirm the available features and the binary target label (`Result`).


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, GridSearchCV
from sklearn.metrics import balanced_accuracy_score

RANDOM_STATE = 42

data_path = Path('AndroidmalwareSmall.xlsx')
df = pd.read_excel(data_path)
print(f"Dataset shape: {df.shape}")
print("Columns:\n", df.columns.tolist()[:10], '...')
print("Target distribution:\n", df['Result'].value_counts())


Dataset shape: (10000, 87)
Columns:
 ['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES'] ...
Target distribution:
 Result
0    5044
1    4956
Name: count, dtype: int64


In [2]:

X = df.drop(columns=['Result'])
y = df['Result']

print(f"Number of features: {X.shape[1]}")
print(f"Positive class proportion: {y.mean():.3f}")


Number of features: 86
Positive class proportion: 0.496



## 2. Model Selection and Generalization Error Estimation

Three repeated stratified hold-outs are used. For each split, the training portion undergoes a grid search (3-fold stratified CV) over AdaBoost hyperparameters, and the tuned model is evaluated on the held-out test data. This yields an unbiased estimate of balanced accuracy (and thus BER) across the three repetitions.


In [3]:

param_grid = {
    'n_estimators': [50, 150],
    'learning_rate': [0.5, 1.0],
    'estimator__max_depth': [1, 2],
}

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
sss = StratifiedShuffleSplit(n_splits=3, test_size=0.2, random_state=RANDOM_STATE + 1)

balanced_accuracies = []
best_params_per_split = []

for split_id, (train_idx, test_idx) in enumerate(sss.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    search = GridSearchCV(
        estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE),
        param_grid=param_grid,
        scoring='balanced_accuracy',
        cv=inner_cv,
        n_jobs=1,
    )
    search.fit(X_train, y_train)
    best_params_per_split.append(search.best_params_)

    y_pred = search.predict(X_test)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    balanced_accuracies.append(bal_acc)
    print(f"Split {split_id} balanced accuracy: {bal_acc:.4f} with params {search.best_params_}")

balanced_accuracies = np.array(balanced_accuracies)

mean_bal_acc = balanced_accuracies.mean()
std_bal_acc = balanced_accuracies.std(ddof=1)
sem_bal_acc = std_bal_acc / np.sqrt(len(balanced_accuracies))
ci_low_bal = mean_bal_acc - 1.96 * sem_bal_acc
ci_high_bal = mean_bal_acc + 1.96 * sem_bal_acc

mean_ber = 1 - mean_bal_acc
ci_low_ber = 1 - ci_high_bal
ci_high_ber = 1 - ci_low_bal

print()
print("Balanced accuracies:", np.round(balanced_accuracies, 4))
print("Mean balanced accuracy: {:.4f}".format(mean_bal_acc))
print("Std balanced accuracy: {:.4f}".format(std_bal_acc))
print("95% CI balanced accuracy: ({:.4f}, {:.4f})".format(ci_low_bal, ci_high_bal))
print()
print("Estimated BER: {:.4f}".format(mean_ber))
print("95% CI BER: ({:.4f}, {:.4f})".format(ci_low_ber, ci_high_ber))
print()
print("Best hyperparameters per split:")
for idx, params in enumerate(best_params_per_split, start=1):
    print(f" Split {idx}: {params}")


Split 1 balanced accuracy: 0.9511 with params {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 150}
Split 2 balanced accuracy: 0.9551 with params {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 150}
Split 3 balanced accuracy: 0.9435 with params {'estimator__max_depth': 2, 'learning_rate': 0.5, 'n_estimators': 150}

Balanced accuracies: [0.9511 0.9551 0.9435]
Mean balanced accuracy: 0.9499
Std balanced accuracy: 0.0059
95% CI balanced accuracy: (0.9433, 0.9565)

Estimated BER: 0.0501
95% CI BER: (0.0435, 0.0567)

Best hyperparameters per split:
 Split 1: {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 150}
 Split 2: {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 150}
 Split 3: {'estimator__max_depth': 2, 'learning_rate': 0.5, 'n_estimators': 150}



## 3. Final Model Fit

Refit the tuned AdaBoost model on the complete dataset using the same grid search configuration.


In [4]:

final_search = GridSearchCV(
    estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=inner_cv,
    n_jobs=1,
)
final_search.fit(X, y)

final_best_bal_acc = final_search.best_score_
final_best_ber = 1 - final_best_bal_acc

print("Best hyperparameters on full dataset:", final_search.best_params_)
print("Cross-validated balanced accuracy (inner CV on full data): {:.4f}".format(final_best_bal_acc))
print("Corresponding balanced error rate: {:.4f}".format(final_best_ber))


Best hyperparameters on full dataset: {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 150}
Cross-validated balanced accuracy (inner CV on full data): 0.9574
Corresponding balanced error rate: 0.0426



## 4. Modeling Notes and Generalization Error Estimate

- **Models explored:** AdaBoost ensembles with decision tree base learners while varying the number of estimators (50, 150), learning rates (0.5, 1.0), and tree depth (1–2).
- **Generalization error estimation:** Three repeated stratified 80/20 hold-outs, each with an inner 3-fold grid search, produced the balanced accuracy distribution summarized above. BER is obtained as `1 - balanced_accuracy`.
- **Parameters deviating from scikit-learn defaults:**
  - `n_estimators` tuned beyond the default of 50; the recommended model uses the value shown above.
  - `learning_rate` tuned beyond the default of 1.0.
  - Decision tree base learner depth was tuned instead of relying on the default depth-1 stump.
  - `random_state` fixed at 42 for reproducibility.

The reported BER estimate and its confidence interval reflect the variability across the repeated hold-out evaluations.
